In [ ]:
!pip install -q peft transformers torch pandas tqdm accelerate bitsandbytes

In [2]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from tqdm import tqdm
from scipy.optimize import minimize
from sklearn.metrics import log_loss, accuracy_score
from sklearn.metrics import accuracy_score, f1_score
from scipy.optimize import differential_evolution
from sklearn.metrics import accuracy_score
import json
from pathlib import Path

2026-01-11 17:27:45.983563: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768152466.323860     167 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768152466.422402     167 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
# Configuration
PATHS = {
    'test_csv': "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/test.csv",
    'deberta': "/kaggle/input/kdsh26-deberta-v3-base-fine-tune-model/deberta-v3-base-nli/checkpoint-40",
    'qwen_v2': "/kaggle/input/kdsh26-qwen2-5-7b-instruct-fine-tune-model/qwen2.5-7b-books-lora-cls/checkpoint-10000",
    'qwen_v1': "/kaggle/input/kdsh26-qwen2-5-7b-instruct-fine-t-model-checkpoint/qwen2.5-7b-books-lora-cls/kaggle/working/qwen2.5-7b-books-lora-cls/checkpoint-5000",
    'output_ensemble': "submission.csv"
}

TEST_PATH = "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/test.csv"
MC_CONS_PATH = "/kaggle/input/kdsh26-jsonl-file-characters-2/Jsonl_file_chars/monte_cristo/monte_cristo_constraints_updated_2.jsonl"
CA_CONS_PATH = "/kaggle/input/kdsh26-jsonl-file-characters-2/Jsonl_file_chars/castaways/castaways_constraints_filled.jsonl"

device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
def load_constraints(path):
    mapping = {}
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            key = (obj["book_name"], obj["character"])
            mapping[key] = obj.get("constraints", [])
    return mapping

def constraint_to_sentence(book, char, c):
    dim, val = c["dimension"], c["value"]
    if dim == "health_state":
        return f"In {book}, {char} is {val}."
    elif dim == "family_role":
        return f"In {book}, {char} has family role: {val}."
    elif dim == "role":
        return f"In {book}, {char} is described as {val}."
    elif dim == "geographic_expertise":
        return f"{char} is familiar with {val}."
    elif dim == "criminal_history":
        return f"{char} has criminal history: {val}."
    else:
        return f"{dim}: {val}."

def build_context(book, char, max_cons=6):
    cons = constraints.get((book, char), [])
    return " ".join(
        constraint_to_sentence(book, char, c)
        for c in cons[:max_cons]
    )


In [5]:
mc_constraints = load_constraints(MC_CONS_PATH)
ca_constraints = load_constraints(CA_CONS_PATH)
constraints = {**mc_constraints, **ca_constraints}

In [6]:
test_df = pd.read_csv(TEST_PATH)

In [7]:
test_df["context"] = test_df.apply(
    lambda r: build_context(r["book_name"], r["char"]), axis=1
)

In [8]:
def load_models():
    print("--- Loading DeBERTa ---")
    deb_tk = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
    deb_mdl = AutoModelForSequenceClassification.from_pretrained(PATHS['deberta'], num_labels=2, local_files_only=True).to(device).eval()
    
    print("--- Loading Qwen Base and Adapters ---")
    q_tk = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
    base_model = AutoModelForSequenceClassification.from_pretrained(
        "Qwen/Qwen2.5-7B-Instruct", 
        num_labels=2, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    
    # Load Qwen v2 as primary adapter
    qwen_mdl = PeftModel.from_pretrained(base_model, PATHS['qwen_v2'], adapter_name="v2")
    # Load Qwen v1 as additional adapter to save memory
    qwen_mdl.load_adapter(PATHS['qwen_v1'], adapter_name="v1")
    qwen_mdl.eval()
    
    return (deb_tk, deb_mdl), q_tk, qwen_mdl

def get_probs_fixed(df, tk, mdl, model_type="qwen", adapter_name=None):
    probs_list = []
    cols = df.columns.tolist()

    ctx_col = next((c for c in ['context', 'context_for_inference', 'full_text'] if c in cols), None)
    stmt_col = next((c for c in ['content', 'statement'] if c in cols), None)

    if not ctx_col or not stmt_col:
        raise KeyError(f"Could not find required columns. Available: {cols}")

    print(f"Using '{ctx_col}' as Premise and '{stmt_col}' as Hypothesis.")

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Inference {model_type}"):
        premise = str(row[ctx_col])
        hypothesis = str(row[stmt_col])

        # SAME TEMPLATE AS TRAINING
        if model_type == "deberta":
            text = f"Premise: {premise}\nHypothesis: {hypothesis}"
        else:
            if adapter_name:
                mdl.set_adapter(adapter_name)
            text = f"Premise: {premise}\nHypothesis: {hypothesis}"

        inputs = tk(
            text,
            max_length=512,
            truncation=True,
            return_tensors="pt",
            padding="max_length"
        ).to(device)

        with torch.no_grad():
            logits = mdl(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

        # class 1 = contradict
        probs_list.append(probs[1])

    return np.array(probs_list)


In [9]:
(deb_tk, deb_mdl), q_tk, qwen_mdl = load_models()

--- Loading DeBERTa ---


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


In [10]:
# =========================
# --- EXECUTION (TEST ONLY)
# =========================

# 1. Load test set
test_df = pd.read_csv(PATHS['test_csv'])

# 2. Rebuild context EXACTLY like training / validation
test_df["context"] = test_df.apply(
    lambda r: build_context(r["book_name"], r["char"]),
    axis=1
)

# 3. Get model probabilities (P(contradict))
p_deb = get_probs_fixed(test_df, deb_tk, deb_mdl, "deberta")
p_qw2 = get_probs_fixed(test_df, q_tk, qwen_mdl, "qwen_v2", adapter_name="v2")
p_qw1 = get_probs_fixed(test_df, q_tk, qwen_mdl, "qwen_v1", adapter_name="v1")

Using 'context' as Premise and 'content' as Hypothesis.


Inference deberta: 100%|██████████| 60/60 [00:01<00:00, 38.23it/s]


Using 'context' as Premise and 'content' as Hypothesis.


Inference qwen_v2: 100%|██████████| 60/60 [00:02<00:00, 22.04it/s]


Using 'context' as Premise and 'content' as Hypothesis.


Inference qwen_v1: 100%|██████████| 60/60 [00:02<00:00, 28.06it/s]


In [13]:
# 4. Stack predictions
pred_matrix = np.vstack([p_deb, p_qw2, p_qw1]).T  # shape: [N, 3]

# 5. Apply FIXED ensemble weights
BEST_W = np.array([0.98402039, 0.00670148, 0.00927813])  # Found using the training
final_probs = np.dot(pred_matrix, BEST_W)   

# 6. Threshold → class
final_preds = (final_probs > 0.5).astype(int)

# 7. Map to labels
test_df["label"] = np.where(
    final_preds == 0, "consistent", "contradict"
)


In [15]:
# 8. Save submission
submission = test_df[["id", "label"]]
submission.to_csv("Submission_Ensemble_Qwen_Deberta_Inference.csv", index=False)

print("Submission file created")
print(submission.head())

Submission file created
    id       label
0   95  contradict
1  136  consistent
2   59  consistent
3   60  consistent
4  124  contradict
